# v6 — shrinking the personalized swing values

The generic output's swing model is calibrated; the personalized one is
consistently over-spread — calibration slope 0.935, 0.954 and 0.948 in 2024,
2025 and 2026: its differences between pitches are about 5% larger than what
happens. The likely cause is the hitter priors pushing values slightly too far.

The adjustment pulls the personalized swing value part-way back toward the
generic one, shrinking only what personalization adds:

```
q_swing = q_generic + k · (q_personalized − q_generic)
```

k = 1 keeps the personalized model, k = 0 the generic one. Takes are untouched
— both outputs share the same take model.

**k is estimated honestly.** It has a closed-form least-squares solution on
held-out swings; each held-out season uses a k fitted on the *other two*.

**The rule, fixed before the numbers:** adopt the shrink if paired held-out
swing MSE improves with the interval excluding zero; calibration then checks
it, and the player checks are guardrails. 2026 is checked afterwards with the
saved final models and a k from all of 2023–25 — an out-of-regime evaluation,
not a decider.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import fit_direct, predict_both, V1_FEATURES
from src.decomposition import fit_decomposed

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
SEASONS = range(2021, 2026)          # 2026 is kept out of model selection; not loaded here
PF = E.PERSONALIZED_FOLDS
SHOW, MODEL = E.SHOW, E.MODEL_COLS
ACC_COLS = ['pitches', 'diff_%', 'diff_%_lo', 'diff_%_hi', 'verdict']
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']
PRIORS = ['prior_whiff', 'prior_foul']

def row(run, label, score=DEC.DEFAULT_SCORE):
    return E.harness(run, label, score)['summary']

def paired(a, b, labels):
    return E.paired_accuracy(a, b, labels)[ACC_COLS]

def rounds(run):
    return pd.DataFrame({season: m.rounds for season, m in run.models.items()}).T

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(SEASONS, verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')

# Hitter priors, each season from the two seasons before it (2021 is prior data only).
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
assert pdf[PRIORS].notna().all().all()
print(f'{len(pdf):,} pitches, seasons {sorted(pdf.season.unique())}')

2,827,989 pitches, seasons [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. Held out, 2023–2025

In [3]:
runs = {name: E.run_folds(pdf, feats, PF, fit=fit_decomposed, in_play='classifier')
        for name, feats in {'generic': V3_FEATURES,
                            'personalized': V3_FEATURES + ['hot_zone'] + PRIORS}.items()}
assert np.array_equal(runs['generic'].held.q_take.to_numpy(), runs['personalized'].held.q_take.to_numpy())
print('both outputs share the same take model: yes')

both outputs share the same take model: yes


In [4]:
# k for each held-out season, fitted on the other two; and k from all three (for 2026).
seasons = list(E.HELD_OUT)
k_loso = {s: E.swing_shrink_factor(runs['personalized'], runs['generic'], [t for t in seasons if t != s])
          for s in seasons}
k_all = E.swing_shrink_factor(runs['personalized'], runs['generic'])
pd.Series({**{f'k for {s} (fitted on the others)': k for s, k in k_loso.items()},
           'k from 2023-25 (used for 2026)': k_all}).round(4).to_frame('k')

,k
k for 2023 (fitted on the others),0.7286
k for 2024 (fitted on the others),0.6636
k for 2025 (fitted on the others),0.6656
k from 2023-25 (used for 2026),0.6846


In [5]:
# The deciding test: shrunk against unshrunk personalized, on the same held-out swings.
shrunk = E.shrink_swing(runs['personalized'], runs['generic'], k_loso)
paired(shrunk, runs['personalized'], ('shrunk', 'personalized')).round(3)

pitches  diff_%  diff_%_lo  diff_%_hi        verdict
action season                                                      
take   2023     374424   0.000      0.000      0.000            tie
       2024     366046   0.000      0.000      0.000            tie
       2025     368391   0.000      0.000      0.000            tie
       pooled  1108861   0.000      0.000      0.000            tie
swing  2023     339205  -0.067     -0.085     -0.051  shrunk better
       2024     337782  -0.028     -0.043     -0.012  shrunk better
       2025     336884  -0.030     -0.046     -0.013  shrunk better
       pooled  1013871  -0.042     -0.050     -0.032  shrunk better

In [6]:
pd.concat({'generic': E.calibration_table(runs['generic']),
           'personalized': E.calibration_table(runs['personalized']),
           'personalized, shrunk': E.calibration_table(shrunk)}, names=['model']).xs('swing', level='action').round(3)

slope  slope_lo  slope_hi
model                season                           
generic              2023    1.063     1.040     1.084
                     2024    0.983     0.961     1.005
                     2025    1.004     0.983     1.027
personalized         2023    0.980     0.960     1.002
                     2024    0.935     0.915     0.954
                     2025    0.954     0.934     0.975
personalized, shrunk 2023    1.039     1.018     1.061
                     2024    0.981     0.961     1.001
                     2025    1.001     0.979     1.024

In [7]:
# Where it was most over-spread: swings at pitches the league usually swings at.
p_swing = E.swing_propensity(pdf, V3_FEATURES, PF)
pd.concat({'personalized': E.propensity_calibration(runs['personalized'], p_swing),
           'personalized, shrunk': E.propensity_calibration(shrunk, p_swing)},
          names=['model']).xs('swing', level='action').round(4)

pitches  mean pred  mean obs   slope  slope_lo  slope_hi
model                league swing propensity                                                          
personalized         (-0.001, 0.1]              13481    -0.0619   -0.0666  0.9228    0.8774    0.9962
                     (0.1, 0.3]                 70228    -0.0520   -0.0515  1.0201    0.9883    1.0481
                     (0.3, 0.7]                356086    -0.0310   -0.0313  0.9815    0.9590    1.0015
                     (0.7, 0.9]                322603    -0.0209   -0.0200  0.9219    0.8958    0.9489
                     (0.9, 1.0]                251473     0.0000   -0.0005  0.9183    0.8833    0.9465
personalized, shrunk (-0.001, 0.1]              13481    -0.0619   -0.0666  0.9497    0.9036    1.0153
                     (0.1, 0.3]                 70228    -0.0521   -0.0515  1.0385    1.0063    1.0680
                     (0.3, 0.7]                356086    -0.0311   -0.0313  1.0200    0.9993    1.0383
                     (0.7, 0.9]                322603    -0.0213   -0.0200  0.9907    0.9612    1.0164
                     (0.9, 1.0]                251473    -0.0008   -0.0005  0.9919    0.9538    1.0208

In [8]:
# Guardrails: the scorecard rows, and shrunk minus unshrunk with hitter-resampled intervals.
display(pd.DataFrame([row(runs['generic'], 'generic'), row(runs['personalized'], 'personalized'),
                      row(shrunk, 'personalized, shrunk')]).set_index('variant')[MODEL + SHOW])
E.metric_differences(shrunk, runs['personalized'], labels=('shrunk', 'personalized')).round(3)

,swing MSE vs count-only %,swing pred slope,take MSE vs count-only %,take pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
generic,1.96,1.017,70.08,0.997,-0.910,0.689,0.816,0.583,0.276,0.094
personalized,2.12,0.956,70.08,0.997,-0.864,0.576,0.844,0.606,0.198,0.159
"personalized, shrunk",2.16,1.007,70.08,0.997,-0.888,0.627,0.833,0.598,0.224,0.144


,shrunk,personalized,diff,diff_lo,diff_hi
check,,,,,
chase | zone_swing,-0.888,-0.864,-0.023,-0.028,-0.020
zone_swing | chase,0.627,0.576,0.051,0.041,0.060
YoY R2 (mean),0.598,0.606,-0.009,-0.015,-0.001
Zone% |r|,0.224,0.198,0.026,0.019,0.032
next-season r (partial),0.144,0.159,-0.016,-0.023,-0.009


In [9]:
# Contamination screens on the shrunk output, and how close it stays to the generic one.
display(pd.DataFrame([{'output': n, **E.zone_contamination(r.held, DEC.DEFAULT_SCORE)}
                      for n, r in [('generic', runs['generic']), ('personalized', runs['personalized']),
                                   ('personalized, shrunk', shrunk)]]).set_index('output').round(3))
sg, ss = (E.player_metric(r.held, DEC.DEFAULT_SCORE) for r in (runs['generic'], shrunk))
m = sg.merge(ss, on=['season', 'batter'])
print(f"generic vs shrunk personalized hitter-season scores: r = {np.corrcoef(m.decision_value_x, m.decision_value_y)[0, 1]:.3f}")

,Zone% r raw,Zone% r | correct-decision rate,"Zone% r | chase, zone-swing",Zone% r | production,"corr(Zone%, production)"
output,,,,,
generic,0.276,0.129,-0.044,0.379,-0.225
personalized,0.198,-0.047,-0.193,0.331,-0.225
"personalized, shrunk",0.224,0.009,-0.162,0.350,-0.225


generic vs shrunk personalized hitter-season scores: r = 0.972


**The shrink is adopted.** Keeping about two thirds of what personalization adds
— k = 0.73, 0.66 and 0.67 for 2023, 2024 and 2025, each fitted on the other two
seasons — makes the personalized swing model more accurate in every held-out
season (−0.067%, −0.028%, −0.030%), pooled −0.042% (interval −0.050% to
−0.032%). The stability of k across seasons is what the earlier recalibration
maps lacked.

**It fixes the calibration.** Swing slopes go from 0.980 / 0.935 / 0.954 to
1.039 / 0.981 / 1.001, and the over-spread on swings at pitches the league
usually swings at disappears (0.92 → 0.99 in the two highest-propensity bands).
The extrapolation bias on rarely-swung pitches is unchanged (−0.062 predicted
against −0.067 observed): it is shared with the generic model, which the shrink
pulls toward.

**The guardrails move both ways.** Construct validity improves
(−0.864 / +0.576 → −0.888 / +0.627) — personalization no longer pulls the score
quite so far toward contact skill — while YoY R² (0.606 → 0.598), Zone%
contamination (0.198 → 0.224) and next-season validity (0.159 → 0.144) give a
little back. All move outside their intervals; none decides.

On the contamination screens the shrunk output sits between the two it blends —
Zone% r 0.224 raw and 0.350 holding production fixed, against 0.198 and 0.331
unshrunk and 0.276 and 0.379 generic — and its hitter-season scores correlate
0.972 with the generic output's.

The shrunk personalized model is the most accurate found: swing MSE 2.16% below
the count-only predictor, against 2.12% unshrunk and 1.96% generic.

## 2. 2026, with the saved final models

The final models from `final_models.ipynb`, reloaded, with k from all of
2023–25. 2026 was inspected by earlier versions, so this is an out-of-regime
evaluation; it does not decide anything.

In [10]:
from src.decomposition import load_models
from src.baselines import predict_both
from pathlib import Path
MODEL_DIR = Path('..') / 'models'
SCORE_SEASON = 2026
TRAIN = {'generic': [2021, 2022, 2023, 2024, 2025], 'personalized': [2022, 2023, 2024, 2025]}
FEATURES = {'generic': V3_FEATURES, 'personalized': V3_FEATURES + ['hot_zone'] + PRIORS}

df = D.drop_pitchers_batting(D.load_seasons(range(2021, SCORE_SEASON + 1), verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
frames = {'generic': df, 'personalized': pdf}       # as in final_models.ipynb; pitches pair on the key

In [11]:
scored = {}
for name in FEATURES:
    feats = FEATURES[name]
    model = load_models(MODEL_DIR / name)
    rv = pd.read_parquet(MODEL_DIR / name / 'run_value_table.parquet')['run_value']
    frame = frames[name].dropna(subset=[f for f in feats if f != 'count'])
    train = D.apply_run_value(frame[frame.season.isin(TRAIN[name])], rv, name='target')
    count_only = train.groupby(['swing', 'count'], observed=True)['target'].mean()
    s26 = D.apply_run_value(frame[frame.season == SCORE_SEASON], rv, name='target').dropna(subset=['target'])
    s26 = DEC.add_scores(predict_both(s26, model)).join(model.components(s26))
    s26['count_only'] = count_only.reindex(pd.MultiIndex.from_arrays([s26['swing'], s26['count']])).to_numpy()
    scored[name] = E.FoldRun(held=s26, models={SCORE_SEASON: model}, folds=())
{name: f'{len(r.held):,} pitches' for name, r in scored.items()}

{'generic': '711,016 pitches', 'personalized': '711,016 pitches'}

In [12]:
shrunk26 = E.shrink_swing(scored['personalized'], scored['generic'], k_all)
display(E.paired_accuracy(shrunk26, scored['personalized'], ('shrunk', 'personalized'))[ACC_COLS].round(3))
pd.concat({'personalized': E.calibration_table(scored['personalized']),
           'personalized, shrunk': E.calibration_table(shrunk26)}, names=['model']).round(3)

pitches  diff_%  diff_%_lo  diff_%_hi        verdict
action season                                                      
take   2026     371986    0.00      0.000      0.000            tie
       pooled   371986    0.00      0.000      0.000            tie
swing  2026     339030   -0.06     -0.076     -0.045  shrunk better
       pooled   339030   -0.06     -0.076     -0.045  shrunk better

slope  slope_lo  slope_hi
model                action season                           
personalized         take   2026    1.016     1.013     1.020
                     swing  2026    0.948     0.924     0.970
personalized, shrunk take   2026    1.016     1.013     1.020
                     swing  2026    0.996     0.973     1.017

**2026 agrees.** With k = 0.685 from 2023–25, the shrunk personalized model is
0.060% more accurate on 2026 swings (interval 0.045–0.076%) and calibrated
(slope 0.996, interval 0.973–1.017, against 0.948 unshrunk). This is an
out-of-regime check, not the evidence for adopting it.

## Findings

**The personalized output keeps about two thirds of what personalization adds.**
Its swing value becomes `q_generic + k · (q_personalized − q_generic)` with
k ≈ 0.68, estimated on held-out swings. That makes it more accurate in every
held-out season (pooled −0.042% of MSE, interval excluding zero) and on 2026
(−0.060%), and it fixes the over-spread: swing calibration slopes 0.98–1.04 held
out and 0.996 on 2026, against 0.935–0.980 and 0.948 before.

The hitter priors were right in direction and too strong in size — the same
lesson as shrinking each hitter's hot zone toward the league, applied one level
up. Construct validity improves; reliability, contamination and next-season
validity give a little back.